In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import sys
sys.path.append('..')
warnings.filterwarnings('ignore')

# Import your classes
from src.data import Data
from src.features import Features
from src.visualization import Visualization
from src.correlation import Correlation

# Initialize classes
data_loader = Data(fs=700)
feature_extractor = Features(fs=700)
viz = Visualization()
corr = Correlation()

## 1. Load the dataset

In [2]:
ecgs,labels = data_loader.read_dataset("../data/WESAD")



## 2. Change the label for the dataset

In [3]:
# Filter to keep only labels 1, 2, 3, 4
KEEP_LABELS = {1, 2, 3, 4}

# Define chunk durations in seconds
chunk_durations = [30, 120, 300]

# Dictionary to store datasets for each chunk duration
datasets = {}

# Process each chunk duration
for duration in chunk_durations:
    all_chunks = []
    all_labels = []
    
    # Process each subject's ECG and labels
    for subj_idx, (ecg, label) in enumerate(zip(ecgs, labels)):
        # Filter for only the labels we want to keep
        mask = np.isin(label, list(KEEP_LABELS))
        filtered_indices = np.where(mask)[0]
        
        if len(filtered_indices) == 0:
            continue
        
        # Get filtered ECG and labels
        filtered_ecg = ecg[mask]
        filtered_label = label[mask]
        
        # Get chunks of specified duration
        chunk_size = data_loader.fs * duration
        
        # Create chunks and track their labels
        for start_idx in range(0, len(filtered_ecg) - chunk_size + 1, chunk_size):
            chunk = filtered_ecg[start_idx:start_idx + chunk_size]
            chunk_labels = filtered_label[start_idx:start_idx + chunk_size]
            
            if len(chunk) == chunk_size:  # Only keep full-length chunks
                all_chunks.append(chunk)
                # Use the most common label in the chunk (should be consistent)
                chunk_label = np.bincount(chunk_labels).argmax()
                all_labels.append(chunk_label)
    
    # Store the dataset
    datasets[duration] = {
        'chunks': np.array(all_chunks),
        'labels': np.array(all_labels)
    }
    
    print(f"Duration {duration}s: {len(all_chunks)} chunks")
    if len(all_labels) > 0:
        unique_labels, counts = np.unique(all_labels, return_counts=True)
        print(f"  Label distribution: {dict(zip(unique_labels, counts))}")

print("\nDatasets created successfully!")
print(f"Available durations: {list(datasets.keys())} seconds")

Duration 30s: 196 chunks
  Label distribution: {np.int64(1): np.int64(77), np.int64(2): np.int64(44), np.int64(3): np.int64(24), np.int64(4): np.int64(51)}
Duration 120s: 49 chunks
  Label distribution: {np.int64(1): np.int64(20), np.int64(2): np.int64(11), np.int64(3): np.int64(6), np.int64(4): np.int64(12)}
Duration 300s: 19 chunks
  Label distribution: {np.int64(1): np.int64(8), np.int64(2): np.int64(4), np.int64(3): np.int64(2), np.int64(4): np.int64(5)}

Datasets created successfully!
Available durations: [30, 120, 300] seconds


## 3. Get features

In [5]:
# Extract features for each dataset and create training-ready dataframes

# Define binary labels: 0 = No Stress (baseline + meditation), 1 = Stress (amusement + stress)
# Label mapping: 1=Baseline, 2=Amusement, 3=Meditation, 4=Stress
# 0 (No Stress): 1, 3
# 1 (Stress): 2, 4

def create_binary_label(original_label):
    """Convert multi-class labels to binary stress classification"""
    if original_label in [1, 3]:  # Baseline, Meditation
        return 0
    else:  # Amusement, Stress
        return 1

# Dictionary to store feature dataframes for each duration
feature_dataframes = {}

# Process each dataset
for duration in chunk_durations:
    print(f"\n{'='*60}")
    print(f"Extracting features for {duration}s chunks...")
    print(f"{'='*60}")
    
    chunks = datasets[duration]['chunks']
    original_labels = datasets[duration]['labels']
    
    # Extract features from all chunks
    all_features = []
    binary_labels = []
    
    for idx, chunk in enumerate(chunks):
        if idx % 20 == 0:
            print(f"Processing chunk {idx+1}/{len(chunks)}")
        
        try:
            # Extract HRV features
            features_dict = feature_extractor.get_hrv_features(chunk)
            features_dict['chunk_index'] = idx
            features_dict['original_label'] = original_labels[idx]
            
            # Add binary stress label
            binary_label = create_binary_label(original_labels[idx])
            features_dict['stress'] = binary_label
            
            all_features.append(features_dict)
            binary_labels.append(binary_label)
        except Exception as e:
            print(f"Error processing chunk {idx}: {e}")
            continue
    
    # Create dataframe
    df = pd.DataFrame(all_features)
    
    # Fill NaN values with 0 for frequency-domain features
    df = df.fillna(0)
    
    print(f"\nDataset statistics for {duration}s:")
    print(f"  Total chunks: {len(df)}")
    
    # Display label distribution
    stress_dist = df['stress'].value_counts().sort_index()
    print(f"\nBinary classification distribution:")
    print(f"  No Stress (0): {stress_dist.get(0, 0)}")
    print(f"  Stress (1): {stress_dist.get(1, 0)}")
    
    # Display feature columns
    feature_cols = [col for col in df.columns if col not in ['chunk_index', 'original_label', 'stress']]
    print(f"\nFeatures extracted: {len(feature_cols)}")
    print(f"  {feature_cols}")
    
    # Store dataframe
    feature_dataframes[duration] = df

print("\n" + "="*60)
print("Feature extraction completed successfully!")
print("="*60)


Extracting features for 30s chunks...
Processing chunk 1/196
Processing chunk 21/196
Processing chunk 41/196
Processing chunk 61/196
Processing chunk 81/196
Processing chunk 101/196
Processing chunk 121/196
Processing chunk 141/196
Processing chunk 161/196
Processing chunk 181/196

Dataset statistics for 30s:
  Total chunks: 196

Binary classification distribution:
  No Stress (0): 101
  Stress (1): 95

Features extracted: 8
  ['mean_rr', 'mean_hr', 'sdnn', 'rmssd', 'pnn50', 'lf_power', 'hf_power', 'lf_hf_ratio']

Extracting features for 120s chunks...
Processing chunk 1/49
Processing chunk 21/49
Processing chunk 41/49

Dataset statistics for 120s:
  Total chunks: 49

Binary classification distribution:
  No Stress (0): 26
  Stress (1): 23

Features extracted: 8
  ['mean_rr', 'mean_hr', 'sdnn', 'rmssd', 'pnn50', 'lf_power', 'hf_power', 'lf_hf_ratio']

Extracting features for 300s chunks...
Processing chunk 1/19

Dataset statistics for 300s:
  Total chunks: 19

Binary classification di

## 4. Intailize the ML models

In [12]:
# Import ML libraries
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Initialize ML Models
print("="*70)
print("INITIALIZING ML MODELS")
print("="*70)

ml_models = {
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'SVM': SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0)
}

print("\nModels initialized:")
for model_name, model in ml_models.items():
    print(f"  ✓ {model_name}: {type(model).__name__}")

print("\n" + "="*70)
print("Ready to train models on datasets!")
print("="*70)

INITIALIZING ML MODELS

Models initialized:
  ✓ KNN: KNeighborsClassifier
  ✓ SVM: SVC
  ✓ Decision Tree: DecisionTreeClassifier
  ✓ Random Forest: RandomForestClassifier
  ✓ Gradient Boosting: GradientBoostingClassifier
  ✓ Logistic Regression: LogisticRegression
  ✓ XGBoost: XGBClassifier

Ready to train models on datasets!


## 5. Train model for each chunking

In [13]:
# Perform cross-validation for each model on each dataset (5-Fold Stratified K-Fold)

cv_results = {}

print("="*80)
print("PERFORMING 5-FOLD STRATIFIED CROSS-VALIDATION")
print("="*80)

for duration in chunk_durations:
    print(f"\n{'='*80}")
    print(f"Cross-Validation for {duration}s dataset")
    print(f"{'='*80}")
    
    # Get the dataset
    df = feature_dataframes[duration]
    
    # Prepare features and labels
    feature_cols = [col for col in df.columns if col not in ['chunk_index', 'original_label', 'stress']]
    X = df[feature_cols].values
    y = df['stress'].values
    
    # Normalize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"\nDataset info:")
    print(f"  Total samples: {len(df)}")
    print(f"  Features: {len(feature_cols)}")
    print(f"  Class distribution: No Stress={np.sum(y==0)}, Stress={np.sum(y==1)}")
    
    # Initialize dictionary for this duration
    cv_results[duration] = {}
    
    # Set up 5-fold stratified cross-validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Perform cross-validation for each model
    for model_name, model in ml_models.items():
        print(f"\n  {model_name}...")
        
        try:
            # Perform cross-validation
            cv_scores_acc = cross_val_score(model, X_scaled, y, cv=skf, scoring='accuracy')
            cv_scores_f1 = cross_val_score(model, X_scaled, y, cv=skf, scoring='f1')
            cv_scores_precision = cross_val_score(model, X_scaled, y, cv=skf, scoring='precision')
            cv_scores_recall = cross_val_score(model, X_scaled, y, cv=skf, scoring='recall')
            
            # Store CV results
            cv_results[duration][model_name] = {
                'accuracy': cv_scores_acc,
                'f1': cv_scores_f1,
                'precision': cv_scores_precision,
                'recall': cv_scores_recall,
                'accuracy_mean': cv_scores_acc.mean(),
                'accuracy_std': cv_scores_acc.std(),
                'f1_mean': cv_scores_f1.mean(),
                'f1_std': cv_scores_f1.std(),
                'precision_mean': cv_scores_precision.mean(),
                'precision_std': cv_scores_precision.std(),
                'recall_mean': cv_scores_recall.mean(),
                'recall_std': cv_scores_recall.std(),
            }
            
            print(f"    ✓ Accuracy:  {cv_scores_acc.mean():.4f} ± {cv_scores_acc.std():.4f}")
            print(f"      F1-Score:  {cv_scores_f1.mean():.4f} ± {cv_scores_f1.std():.4f}")
            
        except Exception as e:
            print(f"    ✗ Error in cross-validation: {e}")

print("\n" + "="*80)
print("CROSS-VALIDATION SUMMARY")
print("="*80)

for duration in chunk_durations:
    print(f"\n{duration}s Dataset:")
    print(f"{'Model':<25} {'Accuracy (±std)':<20} {'F1-Score (±std)':<20}")
    print("-" * 70)
    for model_name in ml_models.keys():
        if model_name in cv_results[duration]:
            results = cv_results[duration][model_name]
            print(f"{model_name:<25} {results['accuracy_mean']:.4f} ± {results['accuracy_std']:.4f}       {results['f1_mean']:.4f} ± {results['f1_std']:.4f}")

PERFORMING 5-FOLD STRATIFIED CROSS-VALIDATION

Cross-Validation for 30s dataset

Dataset info:
  Total samples: 196
  Features: 8
  Class distribution: No Stress=101, Stress=95

  KNN...
    ✓ Accuracy:  0.7349 ± 0.0691
      F1-Score:  0.7014 ± 0.1022

  SVM...
    ✓ Accuracy:  0.7195 ± 0.0969
      F1-Score:  0.6469 ± 0.1545

  Decision Tree...
    ✓ Accuracy:  0.7195 ± 0.0601
      F1-Score:  0.7158 ± 0.0700

  Random Forest...
    ✓ Accuracy:  0.7967 ± 0.0959
      F1-Score:  0.7842 ± 0.1034

  Gradient Boosting...
    ✓ Accuracy:  0.7450 ± 0.0623
      F1-Score:  0.7244 ± 0.0805

  Logistic Regression...
    ✓ Accuracy:  0.7195 ± 0.0643
      F1-Score:  0.6810 ± 0.0743

  XGBoost...
    ✓ Accuracy:  0.7505 ± 0.0873
      F1-Score:  0.7303 ± 0.1009

Cross-Validation for 120s dataset

Dataset info:
  Total samples: 49
  Features: 8
  Class distribution: No Stress=26, Stress=23

  KNN...
    ✓ Accuracy:  0.7733 ± 0.1373
      F1-Score:  0.7543 ± 0.1561

  SVM...
    ✓ Accuracy:  0.71

## 6. Test the models (cross validation)

In [14]:
# Placeholder - visualization will be done in next cell
print("Cross-validation results ready for visualization")

Cross-validation results ready for visualization


## 7. Save figure sof results

In [15]:
# Generate comprehensive comparison figures

output_dir = "results_figures"
os.makedirs(output_dir, exist_ok=True)

print("="*80)
print("GENERATING COMPREHENSIVE MODEL COMPARISON FIGURES")
print("="*80)

# 1. Create comparison table for each duration
print("\n1. Creating per-duration comparison tables...")
for duration in chunk_durations:
    comparison_rows = []
    for model_name in ml_models.keys():
        if model_name in cv_results[duration]:
            results = cv_results[duration][model_name]
            comparison_rows.append({
                'Model': model_name,
                'Accuracy (mean)': f'{results["accuracy_mean"]:.4f}',
                'Accuracy (std)': f'{results["accuracy_std"]:.4f}',
                'F1-Score (mean)': f'{results["f1_mean"]:.4f}',
                'F1-Score (std)': f'{results["f1_std"]:.4f}',
                'Precision (mean)': f'{results["precision_mean"]:.4f}',
                'Recall (mean)': f'{results["recall_mean"]:.4f}',
            })
    
    comparison_df = pd.DataFrame(comparison_rows)
    csv_path = os.path.join(output_dir, f"cv_results_{duration}s.csv")
    comparison_df.to_csv(csv_path, index=False)
    print(f"   Saved: {csv_path}")

# 2. Cross-dataset comparison (all models across all datasets)
print("\n2. Creating cross-dataset comparison figures...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Model Performance Comparison Across Datasets', fontsize=16, fontweight='bold')

# 2.1 Accuracy comparison
ax = axes[0, 0]
for duration in chunk_durations:
    models = list(ml_models.keys())
    accuracies = [cv_results[duration][m]['accuracy_mean'] for m in models]
    errors = [cv_results[duration][m]['accuracy_std'] for m in models]
    ax.errorbar(models, accuracies, yerr=errors, marker='o', label=f'{duration}s', linewidth=2, markersize=6, capsize=5)

ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Accuracy Across All Models', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticklabels(models, rotation=45, ha='right')
ax.set_ylim([0, 1])

# 2.2 F1-Score comparison
ax = axes[0, 1]
for duration in chunk_durations:
    models = list(ml_models.keys())
    f1_scores = [cv_results[duration][m]['f1_mean'] for m in models]
    errors = [cv_results[duration][m]['f1_std'] for m in models]
    ax.errorbar(models, f1_scores, yerr=errors, marker='s', label=f'{duration}s', linewidth=2, markersize=6, capsize=5)

ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title('F1-Score Across All Models', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticklabels(models, rotation=45, ha='right')
ax.set_ylim([0, 1])

# 2.3 Heatmap - Accuracy
ax = axes[1, 0]
accuracy_matrix = pd.DataFrame({
    duration: [cv_results[duration][m]['accuracy_mean'] for m in ml_models.keys()]
    for duration in chunk_durations
}, index=ml_models.keys())

sns.heatmap(accuracy_matrix, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1, ax=ax, cbar_kws={'label': 'Accuracy'})
ax.set_title('Accuracy Heatmap (CV Mean)', fontsize=13, fontweight='bold')
ax.set_xlabel('Dataset Duration (seconds)', fontsize=11)
ax.set_ylabel('Model', fontsize=11)

# 2.4 Heatmap - F1-Score
ax = axes[1, 1]
f1_matrix = pd.DataFrame({
    duration: [cv_results[duration][m]['f1_mean'] for m in ml_models.keys()]
    for duration in chunk_durations
}, index=ml_models.keys())

sns.heatmap(f1_matrix, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1, ax=ax, cbar_kws={'label': 'F1-Score'})
ax.set_title('F1-Score Heatmap (CV Mean)', fontsize=13, fontweight='bold')
ax.set_xlabel('Dataset Duration (seconds)', fontsize=11)
ax.set_ylabel('Model', fontsize=11)

plt.tight_layout()
fig.savefig(os.path.join(output_dir, 'model_comparison_all_metrics.png'), dpi=300, bbox_inches='tight')
print("   Saved: model_comparison_all_metrics.png")
plt.close()

# 3. Per-model comparison across datasets
print("\n3. Creating per-model comparison across datasets...")
for model_name in ml_models.keys():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name} Performance Across Dataset Durations', fontsize=14, fontweight='bold')
    
    durations = chunk_durations
    accuracies = [cv_results[d][model_name]['accuracy_mean'] for d in durations]
    f1_scores = [cv_results[d][model_name]['f1_mean'] for d in durations]
    acc_stds = [cv_results[d][model_name]['accuracy_std'] for d in durations]
    f1_stds = [cv_results[d][model_name]['f1_std'] for d in durations]
    
    # Accuracy
    ax = axes[0]
    ax.bar(range(len(durations)), accuracies, yerr=acc_stds, capsize=5, color='steelblue', alpha=0.7, edgecolor='black')
    ax.set_xticks(range(len(durations)))
    ax.set_xticklabels([f'{d}s' for d in durations])
    ax.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
    ax.set_title('Cross-Validation Accuracy', fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.grid(True, alpha=0.3, axis='y')
    
    # F1-Score
    ax = axes[1]
    ax.bar(range(len(durations)), f1_scores, yerr=f1_stds, capsize=5, color='coral', alpha=0.7, edgecolor='black')
    ax.set_xticks(range(len(durations)))
    ax.set_xticklabels([f'{d}s' for d in durations])
    ax.set_ylabel('F1-Score', fontsize=11, fontweight='bold')
    ax.set_title('Cross-Validation F1-Score', fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    safe_model = model_name.replace(" ", "_").lower()
    fig.savefig(os.path.join(output_dir, f'{safe_model}_across_durations.png'), dpi=300, bbox_inches='tight')
    plt.close()

print("   Saved: individual model comparison figures")

# 4. Summary statistics table
print("\n4. Creating summary statistics table...")
summary_data = []
for duration in chunk_durations:
    for model_name in ml_models.keys():
        if model_name in cv_results[duration]:
            r = cv_results[duration][model_name]
            summary_data.append({
                'Duration': f'{duration}s',
                'Model': model_name,
                'Accuracy': f'{r["accuracy_mean"]:.4f} ± {r["accuracy_std"]:.4f}',
                'F1-Score': f'{r["f1_mean"]:.4f} ± {r["f1_std"]:.4f}',
                'Precision': f'{r["precision_mean"]:.4f} ± {r["precision_std"]:.4f}',
                'Recall': f'{r["recall_mean"]:.4f} ± {r["recall_std"]:.4f}',
            })

summary_df = pd.DataFrame(summary_data)
summary_csv = os.path.join(output_dir, 'cv_summary_all_results.csv')
summary_df.to_csv(summary_csv, index=False)
print(f"   Saved: {summary_csv}")

print("\n" + "="*80)
print(f"ALL FIGURES SAVED TO: {output_dir}")
print("="*80)
print("\nGenerated files:")
print(f"  - model_comparison_all_metrics.png")
print(f"  - cv_results_30s.csv, cv_results_120s.csv, cv_results_300s.csv")
print(f"  - {len(ml_models)} individual model comparison figures")
print(f"  - cv_summary_all_results.csv")

GENERATING COMPREHENSIVE MODEL COMPARISON FIGURES

1. Creating per-duration comparison tables...
   Saved: results_figures\cv_results_30s.csv
   Saved: results_figures\cv_results_120s.csv
   Saved: results_figures\cv_results_300s.csv

2. Creating cross-dataset comparison figures...
   Saved: model_comparison_all_metrics.png

3. Creating per-model comparison across datasets...
   Saved: individual model comparison figures

4. Creating summary statistics table...
   Saved: results_figures\cv_summary_all_results.csv

ALL FIGURES SAVED TO: results_figures

Generated files:
  - model_comparison_all_metrics.png
  - cv_results_30s.csv, cv_results_120s.csv, cv_results_300s.csv
  - 7 individual model comparison figures
  - cv_summary_all_results.csv


In [16]:

# Final Summary and Key Findings

print("\n" + "="*80)
print("NOTEBOOK REVISION SUMMARY")
print("="*80)

print("\n✓ IMPROVEMENTS MADE:")
print("  1. Removed redundant imports (matplotlib.pyplot, seaborn, os duplicates)")
print("  2. Consolidated imports at the beginning of the notebook")
print("  3. Removed test/train split data storage (focused only on cross-validation)")
print("  4. Added 3 new models: Random Forest, Gradient Boosting, Logistic Regression")
print("  5. Generated comprehensive comparison figures:")
print("     - Line plots with error bars (accuracy & F1-Score)")
print("     - Heatmaps for both metrics")
print("     - Per-model comparison across dataset durations")
print("  6. Exported results as CSV for further analysis")

print("\n" + "="*80)
print("KEY FINDINGS - BEST MODELS PER DATASET")
print("="*80)

for duration in chunk_durations:
    print(f"\n{duration}s Dataset:")
    
    # Find best accuracy
    best_acc_model = max(ml_models.keys(), 
                         key=lambda m: cv_results[duration][m]['accuracy_mean'])
    best_acc = cv_results[duration][best_acc_model]['accuracy_mean']
    
    # Find best F1
    best_f1_model = max(ml_models.keys(), 
                        key=lambda m: cv_results[duration][m]['f1_mean'])
    best_f1 = cv_results[duration][best_f1_model]['f1_mean']
    
    print(f"  Best Accuracy:  {best_acc_model:20s} → {best_acc:.4f}")
    print(f"  Best F1-Score:  {best_f1_model:20s} → {best_f1:.4f}")

print("\n" + "="*80)
print("OVERALL BEST MODELS")
print("="*80)

all_results = []
for duration in chunk_durations:
    for model_name in ml_models.keys():
        if model_name in cv_results[duration]:
            r = cv_results[duration][model_name]
            all_results.append({
                'Duration': duration,
                'Model': model_name,
                'Accuracy': r['accuracy_mean'],
                'F1_Score': r['f1_mean'],
            })

results_df = pd.DataFrame(all_results)

print("\nTop 5 by Accuracy:")
top_acc = results_df.nlargest(5, 'Accuracy')[['Duration', 'Model', 'Accuracy']]
for idx, row in top_acc.iterrows():
    print(f"  {row['Duration']:3d}s  {row['Model']:20s}  {row['Accuracy']:.4f}")

print("\nTop 5 by F1-Score:")
top_f1 = results_df.nlargest(5, 'F1_Score')[['Duration', 'Model', 'F1_Score']]
for idx, row in top_f1.iterrows():
    print(f"  {row['Duration']:3d}s  {row['Model']:20s}  {row['F1_Score']:.4f}")

print("\n" + "="*80)
print("NOTEBOOK CLEANUP COMPLETE - READY FOR ANALYSIS")
print("="*80)


NOTEBOOK REVISION SUMMARY

✓ IMPROVEMENTS MADE:
  1. Removed redundant imports (matplotlib.pyplot, seaborn, os duplicates)
  2. Consolidated imports at the beginning of the notebook
  3. Removed test/train split data storage (focused only on cross-validation)
  4. Added 3 new models: Random Forest, Gradient Boosting, Logistic Regression
  5. Generated comprehensive comparison figures:
     - Line plots with error bars (accuracy & F1-Score)
     - Heatmaps for both metrics
     - Per-model comparison across dataset durations
  6. Exported results as CSV for further analysis

KEY FINDINGS - BEST MODELS PER DATASET

30s Dataset:
  Best Accuracy:  Random Forest        → 0.7967
  Best F1-Score:  Random Forest        → 0.7842

120s Dataset:
  Best Accuracy:  KNN                  → 0.7733
  Best F1-Score:  KNN                  → 0.7543

300s Dataset:
  Best Accuracy:  Random Forest        → 0.7833
  Best F1-Score:  Gradient Boosting    → 0.6533

OVERALL BEST MODELS

Top 5 by Accuracy:
   30s